# 02. Primera comparacion de modelos de clasificacion

## Objetivo de esta fase

En esta etapa se construye una primera comparacion entre varios clasificadores vistos en clase para predecir `delay_15`, donde `delay_15 = 1` si `ARR_DELAY >= 15` y `0` en caso contrario.

Datasets utilizados:
- `data/samples/train_sample_2023.parquet` para experimentacion inicial.
- `data/final/valid_2023.parquet` para validacion temporal.

Puntos metodologicos que se mantienen fijos:
- No se usa `test_2024` en esta fase.
- No se cambia el split temporal ya definido por el proyecto.
- `ARR_DELAY` se excluye de `X` porque es la variable a partir de la cual se construye el target y su uso produciria leakage.

`train_sample_2023` ya es una muestra estratificada del entrenamiento de 2023. Aun asi, como esta primera comparacion incluye `KNeighborsClassifier` y se busca una ejecucion ligera y reproducible, el notebook define por defecto una submuestra fija de benchmarking extraida de `train_sample_2023` y `valid_2023`. Ademas, KNN se ejecuta con una representacion categorial mas compacta y con un subconjunto de features algo mas contenido para evitar que el coste de la one-hot encoding domine toda la fase. Los mejores candidatos deberan reentrenarse mas adelante con mayor volumen antes de pasar al test final.


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

RANDOM_STATE = 42
TARGET_COL = 'delay_15'
LEAKAGE_COL = 'ARR_DELAY'
TRAIN_PATH = Path('data/samples/train_sample_2023.parquet')
VALID_PATH = Path('data/final/valid_2023.parquet')

TRAIN_BENCHMARK_SIZE = 20_000
VALID_BENCHMARK_SIZE = 10_000
KNN_TRAIN_METRIC_SIZE = 3_000
INFREQUENT_CATEGORY_MIN_FREQUENCY = 100
KNN_HIGH_CARDINALITY_DROP_COLS = ['ORIGIN', 'DEST']
KNN_N_JOBS = 1


## 1. Carga de datos

Se cargan los dos artefactos pedidos para esta fase y se verifica su coherencia basica. La validacion sigue siendo temporal porque el conjunto de validacion procede exclusivamente de los meses 10 a 12 de 2023.


In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

assert list(train_df.columns) == list(valid_df.columns), 'Train y valid no tienen las mismas columnas.'
assert train_df['MONTH'].between(1, 9).all(), 'train_sample_2023 debe contener solo meses 1 a 9.'
assert valid_df['MONTH'].between(10, 12).all(), 'valid_2023 debe contener solo meses 10 a 12.'

summary_df = pd.DataFrame(
    {
        'dataset': ['train_sample_2023', 'valid_2023'],
        'shape': [train_df.shape, valid_df.shape],
        'delay_rate': [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean()],
        'months': [sorted(train_df['MONTH'].unique().tolist()), sorted(valid_df['MONTH'].unique().tolist())],
    }
)

display(summary_df)


In [ ]:
def make_benchmark_slice(df, target_col, n_rows=None, random_state=42):
    if n_rows is None or n_rows >= len(df):
        return df.copy()

    benchmark_df, _ = train_test_split(
        df,
        train_size=n_rows,
        stratify=df[target_col],
        random_state=random_state,
    )
    return benchmark_df.sort_index().reset_index(drop=True)

train_benchmark = make_benchmark_slice(train_df, TARGET_COL, TRAIN_BENCHMARK_SIZE, RANDOM_STATE)
valid_benchmark = make_benchmark_slice(valid_df, TARGET_COL, VALID_BENCHMARK_SIZE, RANDOM_STATE)

benchmark_summary = pd.DataFrame(
    {
        'dataset': ['train_benchmark', 'valid_benchmark'],
        'shape': [train_benchmark.shape, valid_benchmark.shape],
        'delay_rate': [train_benchmark[TARGET_COL].mean(), valid_benchmark[TARGET_COL].mean()],
        'source_artifact': [TRAIN_PATH.as_posix(), VALID_PATH.as_posix()],
    }
)

display(benchmark_summary)


## 2. Definicion de target y features

El target es `delay_15`. Para construir la matriz de entrada:
- se excluye `delay_15` por ser la variable objetivo,
- se excluye `ARR_DELAY` por la regla anti-leakage,
- se eliminan columnas constantes en el entrenamiento de benchmarking, ya que no aportan variacion para esta fase,
- para KNN se define ademas una version ligera del conjunto de features que evita `ORIGIN` y `DEST`, dos categoricas de cardinalidad extrema en esta etapa.

En particular, `ARR_DELAY` no puede entrar en `X` porque `delay_15` se deriva directamente de esa variable.


In [ ]:
constant_cols = [
    col for col in train_benchmark.columns
    if train_benchmark[col].nunique(dropna=False) == 1
]

excluded_from_model = sorted(set([TARGET_COL, LEAKAGE_COL] + constant_cols))
feature_cols = [col for col in train_benchmark.columns if col not in excluded_from_model]
knn_feature_cols = [col for col in feature_cols if col not in KNN_HIGH_CARDINALITY_DROP_COLS]

X_train = train_benchmark[feature_cols].copy()
y_train = train_benchmark[TARGET_COL].copy()
X_valid = valid_benchmark[feature_cols].copy()
y_valid = valid_benchmark[TARGET_COL].copy()

feature_audit = pd.DataFrame(
    {
        'group': [
            'excluded_from_model',
            'shared_numeric_features',
            'shared_categorical_features',
            'knn_dropped_high_cardinality',
            'knn_features',
        ],
        'columns': [
            excluded_from_model,
            X_train.select_dtypes(exclude=['object', 'string', 'category']).columns.tolist(),
            X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist(),
            KNN_HIGH_CARDINALITY_DROP_COLS,
            knn_feature_cols,
        ],
    }
)

display(feature_audit)
print(f'Numero de features compartidas: {len(feature_cols)}')
print(f'Numero de features para KNN: {len(knn_feature_cols)}')


## 3. Preprocesado

Se separan variables numericas y categoricas. El flujo no es exactamente el mismo para todos los modelos:
- modelos lineales: imputacion + escalado en numericas, y one-hot encoding con agrupacion de categorias infrecuentes,
- KNN: mismo tipo de preprocesado que los modelos lineales, pero con una representacion mas ligera que excluye `ORIGIN` y `DEST` por su alta cardinalidad,
- modelos basados en arboles: imputacion en numericas, one-hot encoding en categoricas y sin escalado adicional.

Ademas, se usan configuraciones baseline o razonables, sin busqueda exhaustiva de hiperparametros.


In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_features = [col for col in X_train.columns if col not in categorical_features]

categorical_features_knn = [col for col in categorical_features if col not in KNN_HIGH_CARDINALITY_DROP_COLS]
numeric_features_knn = [col for col in knn_feature_cols if col not in categorical_features_knn]

def make_grouped_onehot_encoder(min_frequency=100):
    try:
        return OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=min_frequency,
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)

linear_categorical_encoder = make_grouped_onehot_encoder(INFREQUENT_CATEGORY_MIN_FREQUENCY)
knn_categorical_encoder = make_grouped_onehot_encoder(INFREQUENT_CATEGORY_MIN_FREQUENCY)

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', linear_categorical_encoder),
                ]
            ),
            categorical_features,
        ),
    ]
)

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]
            ),
            numeric_features_knn,
        ),
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', knn_categorical_encoder),
                ]
            ),
            categorical_features_knn,
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
                ]
            ),
            categorical_features,
        ),
    ]
)

model_configs = {
    'DummyClassifier': {
        'pipeline': Pipeline(
            steps=[
                ('preprocess', tree_preprocessor),
                ('model', DummyClassifier(strategy='prior')),
            ]
        ),
        'feature_columns': feature_cols,
        'train_metric_size': None,
    },
    'LogisticRegression': {
        'pipeline': Pipeline(
            steps=[
                ('preprocess', linear_preprocessor),
                ('model', LogisticRegression(max_iter=400, solver='liblinear', class_weight='balanced', random_state=RANDOM_STATE)),
            ]
        ),
        'feature_columns': feature_cols,
        'train_metric_size': None,
    },
    'KNeighborsClassifier': {
        'pipeline': Pipeline(
            steps=[
                ('preprocess', knn_preprocessor),
                ('model', KNeighborsClassifier(n_neighbors=15, weights='distance', p=2, n_jobs=KNN_N_JOBS)),
            ]
        ),
        'feature_columns': knn_feature_cols,
        'train_metric_size': KNN_TRAIN_METRIC_SIZE,
    },
    'DecisionTreeClassifier': {
        'pipeline': Pipeline(
            steps=[
                ('preprocess', tree_preprocessor),
                ('model', DecisionTreeClassifier(min_samples_leaf=50, class_weight='balanced', random_state=RANDOM_STATE)),
            ]
        ),
        'feature_columns': feature_cols,
        'train_metric_size': None,
    },
    'RandomForestClassifier': {
        'pipeline': Pipeline(
            steps=[
                ('preprocess', tree_preprocessor),
                ('model', RandomForestClassifier(
                    n_estimators=100,
                    min_samples_leaf=20,
                    class_weight='balanced_subsample',
                    n_jobs=1,
                    random_state=RANDOM_STATE,
                )),
            ]
        ),
        'feature_columns': feature_cols,
        'train_metric_size': None,
    },
}

model_notes = pd.DataFrame(
    {
        'model': list(model_configs.keys()),
        'comment': [
            'Baseline ingenuo para medir si los demas modelos aportan senal real.',
            'Modelo lineal con class_weight balanced y categorias infrecuentes agrupadas.',
            'KNN con n_neighbors mas simple, categorias raras agrupadas y sin ORIGIN ni DEST para contener el coste computacional.',
            'Arbol individual con regularizacion simple via min_samples_leaf.',
            'Bosque aleatorio moderado, sin tuning exhaustivo y con ajuste ligero al desbalance.',
        ],
    }
)

display(model_notes)


## 4. Entrenamiento y evaluacion

La comparacion se hace sobre la submuestra temporal de validacion definida arriba. Se reportan `accuracy`, `precision`, `recall`, `f1`, `ROC-AUC` y la matriz de confusion. Para detectar senales basicas de overfitting tambien se conserva `f1` en entrenamiento, aunque para KNN ese calculo se hace solo sobre una submuestra fija y reproducible del train para no penalizar en exceso el tiempo de ejecucion.


In [ ]:
def make_train_metric_slice(X, y, sample_size=None, random_state=42):
    if sample_size is None or sample_size >= len(X):
        return X, y, 'full_train'

    X_metric, _, y_metric, _ = train_test_split(
        X,
        y,
        train_size=sample_size,
        stratify=y,
        random_state=random_state,
    )
    return X_metric, y_metric, f'subsample_{sample_size}'

def evaluate_model(model_name, config, X_train, y_train, X_valid, y_valid):
    feature_columns = config['feature_columns']
    pipeline = config['pipeline']

    X_train_model = X_train[feature_columns]
    X_valid_model = X_valid[feature_columns]

    fit_start = time.perf_counter()
    pipeline.fit(X_train_model, y_train)
    fit_seconds = time.perf_counter() - fit_start

    X_train_metric, y_train_metric, train_metric_scope = make_train_metric_slice(
        X_train_model,
        y_train,
        config['train_metric_size'],
        RANDOM_STATE,
    )

    train_pred = pipeline.predict(X_train_metric)
    valid_pred = pipeline.predict(X_valid_model)
    valid_proba = pipeline.predict_proba(X_valid_model)[:, 1] if hasattr(pipeline, 'predict_proba') else None

    f1_train_value = f1_score(y_train_metric, train_pred, zero_division=0)
    f1_valid_value = f1_score(y_valid, valid_pred, zero_division=0)

    metrics = {
        'model': model_name,
        'fit_seconds': fit_seconds,
        'train_metric_scope': train_metric_scope,
        'accuracy_valid': accuracy_score(y_valid, valid_pred),
        'precision_valid': precision_score(y_valid, valid_pred, zero_division=0),
        'recall_valid': recall_score(y_valid, valid_pred, zero_division=0),
        'f1_valid': f1_valid_value,
        'f1_train': f1_train_value,
        'f1_gap_train_valid': f1_train_value - f1_valid_value,
        'roc_auc_valid': roc_auc_score(y_valid, valid_proba) if valid_proba is not None else float('nan'),
        'confusion_matrix_valid': confusion_matrix(y_valid, valid_pred).tolist(),
    }
    return metrics, valid_pred, pipeline

results = []
valid_predictions = {}
trained_models = {}

for model_name, config in model_configs.items():
    metrics, valid_pred, fitted_pipeline = evaluate_model(model_name, config, X_train, y_train, X_valid, y_valid)
    results.append(metrics)
    valid_predictions[model_name] = valid_pred
    trained_models[model_name] = fitted_pipeline

results_df = pd.DataFrame(results).sort_values('f1_valid', ascending=False).reset_index(drop=True)

summary_cols = [
    'model',
    'fit_seconds',
    'train_metric_scope',
    'accuracy_valid',
    'precision_valid',
    'recall_valid',
    'f1_valid',
    'roc_auc_valid',
    'f1_train',
    'f1_gap_train_valid',
]

display(results_df[summary_cols].round(4))


## 5. Matrices de confusion y lectura de la clase positiva

Como la clase positiva (`delay_15 = 1`) es la que mas interesa detectar, no conviene quedarse solo con `accuracy`. Las siguientes matrices y tabla ayudan a ver el trade-off entre `precision` y `recall`.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, model_name in zip(axes, results_df['model']):
    ConfusionMatrixDisplay.from_predictions(
        y_valid,
        valid_predictions[model_name],
        ax=ax,
        colorbar=False,
        values_format=',',
    )
    ax.set_title(model_name)

for ax in axes[len(results_df):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

positive_class_view = results_df[
    ['model', 'precision_valid', 'recall_valid', 'f1_valid', 'roc_auc_valid', 'f1_gap_train_valid']
].round(4)

display(positive_class_view)


## 6. Comentario metodologico

Esta comparacion es util para filtrar candidatos, no para declarar un modelo final del proyecto. Sus principales limites son:
- se apoya en una muestra de entrenamiento para agilizar la experimentacion,
- el benchmark por defecto usa una submuestra reproducible mas pequena para que `KNeighborsClassifier` siga siendo ejecutable,
- KNN trabaja con categorias infrecuentes agrupadas y sin `ORIGIN` ni `DEST`, por lo que su comparacion debe leerse como una primera referencia ligera y no como su mejor version posible,
- aun no hay refinamiento de hiperparametros ni analisis de umbral.

El siguiente paso razonable sera reentrenar los modelos mas prometedores con mas volumen de `train_2023` y volver a contrastarlos contra `valid_2023` antes de tocar `test_2024`.


In [ ]:
best_f1 = results_df.iloc[0]
best_auc = results_df.sort_values('roc_auc_valid', ascending=False).iloc[0]
best_recall = results_df.sort_values('recall_valid', ascending=False).iloc[0]
most_overfit = results_df.sort_values('f1_gap_train_valid', ascending=False).iloc[0]

print(f"Mejor F1 en validacion: {best_f1['model']} ({best_f1['f1_valid']:.4f}).")
print(f"Mayor ROC-AUC en validacion: {best_auc['model']} ({best_auc['roc_auc_valid']:.4f}).")
print(f"Mayor recall sobre la clase positiva: {best_recall['model']} ({best_recall['recall_valid']:.4f}).")
print(f"Mayor brecha train-valid en F1: {most_overfit['model']} ({most_overfit['f1_gap_train_valid']:.4f}).")


## 7. Conclusiones de la fase

La lectura de cierre debe centrarse en tres preguntas:
- que modelos superan claramente al dummy cuando se mira la clase positiva,
- que candidatos merecen una siguiente fase de refinamiento,
- que senales simples de overfitting aparecen ya en esta comparacion inicial.


In [ ]:
non_dummy = results_df[results_df['model'] != 'DummyClassifier'].copy()
clearly_better = non_dummy[non_dummy['f1_valid'] > results_df.loc[results_df['model'] == 'DummyClassifier', 'f1_valid'].iloc[0]]
knn_row = results_df.loc[results_df['model'] == 'KNeighborsClassifier'].iloc[0]
candidate_models = list(dict.fromkeys(results_df.head(2)['model'].tolist()))

print('Modelos que superan claramente al dummy en F1 de validacion:')
for model_name in clearly_better['model'].tolist():
    print(f'- {model_name}')

print('\nLectura breve de esta fase:')
if len(candidate_models) == 1:
    candidate_text = candidate_models[0]
else:
    candidate_text = ' y '.join(candidate_models)

print(
    f"- Los candidatos mas prometedores para una siguiente iteracion son {candidate_text}, "
    'porque combinan mejor senal en la clase positiva y capacidad discriminativa que el baseline ingenuo.'
)
print(
    f"- KNN muestra un F1 de validacion de {knn_row['f1_valid']:.4f}; si queda por detras de los mejores candidatos, "
    'aporta poco valor competitivo en este benchmark inicial.'
)
print(
    f"- La brecha de F1 entre train y valid sugiere vigilar sobreajuste, especialmente en {most_overfit['model']}, "
    'antes de avanzar hacia una fase mas ambiciosa de refinamiento.'
)
